# Demo: recorrido de delegación

Este notebook recorre la traza que dejó `python main.py` en `trazas/ejecucion.json`.
No llama a ninguna API: solo lee el archivo, así que se puede abrir sin keys.

Para cada pregunta muestra qué decidió el supervisor en cada paso, qué hizo cada
especialista y cuál fue la respuesta final.

In [1]:
import json
from pathlib import Path

traza = json.loads(Path("trazas/ejecucion.json").read_text(encoding="utf-8"))
print(f"{len(traza)} preguntas en la traza")

2 preguntas en la traza


## Resumen

Qué agentes participaron en cada pregunta, en orden, y cuánto tardó.

In [2]:
for i, r in enumerate(traza, 1):
    agentes = " → ".join(r["agentes_usados"]) or "ninguno"
    print(f"{i}. {r['demuestra']}")
    print(f"   agentes: {agentes}  ({r['segundos']}s)\n")

1. delegación completa: investigador busca el dato, analista calcula
   agentes: investigador → analista  (38.94s)

2. solo investigación: el supervisor no deriva al analista
   agentes: investigador  (18.78s)



## Recorrido paso a paso

`supervisor` muestra la decisión y la razón que dio (citando la rúbrica). Los
especialistas muestran la instrucción que recibieron, las herramientas que usaron
y lo que devolvieron.

In [3]:
def mostrar(r):
    print("PREGUNTA:", r["pregunta"], "\n")
    for paso in r["recorrido"]:
        nodo = paso["nodo"]
        if nodo == "supervisor":
            print(f"[supervisor, decisión {paso['paso']}] → {paso['siguiente']}")
            print(f"    razón: {paso['razon']}")
            if paso["instruccion"]:
                print(f"    instrucción: {paso['instruccion']}")
        elif nodo == "sintesis":
            print("\n[síntesis] respuesta final:\n")
            print(paso["respuesta"])
        elif nodo == "corte":
            print("[corte]", paso["texto"])
        else:
            herramientas = ", ".join(paso["herramientas"]) or "ninguna"
            print(f"[{nodo}] herramientas: {herramientas}")
            print(f"    resultado: {paso['resultado']}")
        print()

mostrar(traza[0])

PREGUNTA: Un servicio necesita mantener 250 conexiones simultáneas con un httpx.Client creado sin parámetros. ¿Alcanza el límite de conexiones por defecto? Si no, ¿cuántas faltan y qué porcentaje del pedido cubre? 

[supervisor, decisión 1] → investigador
    razón: No hay ninguna contribución aún. El primer paso es obtener el valor del límite de conexiones por defecto de httpx.Client de la documentación oficial (criterio 1 de la rúbrica).
    instrucción: Buscá en la documentación oficial de httpx cuál es el límite máximo de conexiones simultáneas (pool size o connection limit) que usa httpx.Client cuando se instancia sin parámetros. Reportá el valor exacto y el archivo o sección de la documentación donde lo encontraste.

[investigador] herramientas: buscar_documentacion, buscar_documentacion
    resultado: El dato fue encontrado de forma clara y consistente en ambas búsquedas. Acá está el resultado:

---

## Límite máximo de conexiones simultáneas en `httpx.Client()` (sin parámetros)

## Segunda pregunta: solo investigación

Acá el supervisor no tiene nada que calcular, así que no debería derivar al analista.

In [4]:
mostrar(traza[1])

PREGUNTA: ¿Qué excepción lanza httpx cuando no consigue una conexión del pool a tiempo? 

[supervisor, decisión 1] → investigador
    razón: No hay ninguna contribución aún. La pregunta requiere un dato concreto de la documentación de httpx (qué excepción se lanza cuando no se obtiene una conexión del pool a tiempo), lo cual es tarea del investigador.
    instrucción: Busca en la documentación oficial de httpx qué excepción se lanza cuando no se puede obtener una conexión del pool de conexiones a tiempo (timeout esperando una conexión disponible del pool). Indicá el nombre exacto de la excepción y el archivo o sección de la documentación donde aparece.

[investigador] herramientas: buscar_documentacion, buscar_documentacion
    resultado: La documentación es clara y consistente en ambas búsquedas. Aquí está el resultado:

---

## Excepción: `PoolTimeout`

Cuando httpx **no puede obtener una conexión del pool de conexiones dentro del tiempo límite configurado**, lanza la excepción:

###

## Quién aportó qué

Las contribuciones de cada especialista quedan en el estado con el paso en que se
hicieron. Esta tabla las cuenta por agente y por herramienta.

In [5]:
from collections import Counter

for r in traza:
    especialistas = [p for p in r["recorrido"] if p["nodo"] not in ("supervisor", "sintesis", "corte")]
    por_agente = Counter(p["nodo"] for p in especialistas)
    por_herramienta = Counter(h for p in especialistas for h in p["herramientas"])
    decisiones = sum(1 for p in r["recorrido"] if p["nodo"] == "supervisor")
    print(r["demuestra"])
    print(f"   decisiones del supervisor: {decisiones}")
    print(f"   aportes por agente: {dict(por_agente)}")
    print(f"   llamadas por herramienta: {dict(por_herramienta)}\n")

delegación completa: investigador busca el dato, analista calcula
   decisiones del supervisor: 3
   aportes por agente: {'investigador': 1, 'analista': 1}
   llamadas por herramienta: {'buscar_documentacion': 2, 'comparar_con_limite': 1, 'calcular': 2}

solo investigación: el supervisor no deriva al analista
   decisiones del supervisor: 2
   aportes por agente: {'investigador': 1}
   llamadas por herramienta: {'buscar_documentacion': 2}

